In [ ]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import matplotlib.pyplot as plt
import scienceplots
plt.style.use(['science', 'notebook'])
plt.rcParams.update({
    "text.usetex": False,
})

from mg_sampling import PROJECT_DIR, FORCEFIELDS
from mg_sampling.classes import Sim
from mg_sampling.plot import plot_site_global_occupancy, plot_site_heatmap, plot_site_occupancy, plot_bound_ion, LABELS

os.makedirs('figures/replex', exist_ok=True)

In [ ]:
repids = [0]
systems = ['7ez0', '7ez0_noion']
forcefields = ['mMg_tip3p', 'nMg_tip3p', 'mMg_opc']

for forcefield in forcefields:
    for system in systems:
        fig, axarr = plt.subplots(nrows=len(repids), ncols=4, figsize=(16, 4), sharex=True, sharey=True)
        axarr = np.atleast_2d(axarr)
        for i, ax in zip(repids, axarr):
            path = PROJECT_DIR / 'replex' / system / forcefield / f'rep{i:02d}'
            sim = Sim(path, colvar_file=f'bias.COLVAR.{i}')
            _ = plot_bound_ion(sim, ion='MG', site='O_Phosphate', ax=ax[0])
            _ = plot_bound_ion(sim, ion='NA', site='O_Phosphate', ax=ax[1])
            _ = plot_bound_ion(sim, ion='MG', site='O_Base', ax=ax[2])
            _ = plot_bound_ion(sim, ion='NA', site='O_Base', ax=ax[3])
            for _ax in ax:
                _ax.text(0.1, 0.9, f"Rep. {i}", transform=_ax.transAxes)
        axarr[0, 0].legend(fontsize=10)
        plt.gcf().suptitle(f"{LABELS[system]} {LABELS[forcefield]}")
        plt.tight_layout()
        plt.savefig(f'figures/replex/bound_cations_{system}_{forcefield}.pdf')

In [ ]:
repids = [0, 5, 11]
systems = ['7ez0']
forcefields = ['mMg_tip3p', 'nMg_tip3p', 'mMg_opc']

for forcefield in forcefields:
    for system in systems:
        fig, axarr = plt.subplots(nrows=len(repids), ncols=4, figsize=(16, 8))
        for i, ax in zip(repids, axarr):
            path = PROJECT_DIR / 'replex' / system / forcefield / f'rep{i:02d}'
            sim = Sim(path, colvar_file=f'bias.COLVAR.{i}')
            _ = plot_site_occupancy(sim, ion='MG', site='O_Phosphate', ax=ax[0])
            _ = plot_site_occupancy(sim, ion='NA', site='O_Phosphate', ax=ax[1])
            _ = plot_site_occupancy(sim, ion='MG', site='O_Base', ax=ax[2])
            _ = plot_site_occupancy(sim, ion='NA', site='O_Base', ax=ax[3])
            for _ax in ax:
                _ax.text(0.1, 0.9, f"Rep. {i}", transform=_ax.transAxes)
        plt.gcf().suptitle(f"{LABELS[system]} {LABELS[forcefield]}")
        plt.tight_layout()
        plt.savefig(f'figures/replex/site_occupancy_{system}_{forcefield}.pdf')

### Motifs

In [ ]:
repids = [0, 5, 11]

for forcefield in FORCEFIELDS:
    fig, axarr = plt.subplots(nrows=len(repids))
    for i, ax in zip(repids, axarr):
        path = PROJECT_DIR / 'replex' / '7ez0' / forcefield / f'rep{i:02d}'
        try:
            sim = Sim(path, colvar_file=f'bias.COLVAR.{i}')
        except FileNotFoundError as e:
            print(e)
            continue
        ax = plot_site_heatmap(sim, ax=ax)
        ax.set_title(f"{ax.get_title()} {forcefield}")

In [ ]:
root_dir = PROJECT_DIR / 'replex' / 'motifs_hard'

pdbids = ['2r8s', '3q3z']
otypes = ['op', 'ob']
otype_sel = {
    'op': 'O_Phosphate',
    'ob': 'O_Base',
}
ions = ['MG', 'CL']
forcefields = ['mMg_tip3p']

for pdbid in pdbids:
    for forcefield in forcefields:
        for otype in otypes:
            fig, axarr = plt.subplots(nrows=len(ions), ncols=2, figsize=(10, 8))
            for ion, axrow in zip(ions, axarr):
                for noion, ax in zip(['', '_noion'], axrow):
                    path = root_dir / f'{pdbid}{noion}' / forcefield / 'rep03'
                    ax = plot_site_heatmap(path, site=otype_sel[otype], ion=ion, ax=ax)
                    if noion: ax.set_title(ax.get_title() + " (no PDB ions)")
            plt.tight_layout()
            fig.savefig(f'figures/motif_hard_op_ob_{pdbid}_chronogram_{otype}_{ion}_{forcefield}.pdf')

In [ ]:
from mg_sampling.tools import read_plumed


In [ ]:
root_dir = PROJECT_DIR / 'replex' / 'motifs_hard'

path = root_dir / f'{pdbid}{noion}' / forcefield / 'rep03'
colvar = read_plumed(path / 'no.COLVAR.3')
colvar_bias = read_plumed(path / 'bias.COLVAR.3')
for i in range(10):
    nob = colvar[f'cvob.nb-{i}']
    nox = colvar[f'cvob.nc-{i}']
    bias = colvar_bias[f'cvob.func-{i}']
    nox = nox[nob > 0]
    bias = bias[nob > 0]
    nob = nob[nob > 0]
    plt.scatter(nob, nox, c=bias)
plt.show()

root_dir = PROJECT_DIR / 'replex' / 'motifs_hard_op_ob'

path = root_dir / f'{pdbid}{noion}' / forcefield / 'rep03'
colvar_bias = read_plumed(path / 'bias.COLVAR.3')
colvar = read_plumed(path / 'no.COLVAR.3')
for i in range(10):
    nob = colvar[f'cvob.nb-{i}']
    nox = colvar[f'cvob.nc-{i}']
    bias = colvar_bias[f'cvob.func-{i}']
    nox = nox[nob > 0]
    bias = bias[nob > 0]
    nob = nob[nob > 0]
    plt.scatter(nob, nox, c=bias)
plt.show()

In [ ]:
pdbid = '7ez0'
otypes = ['op', 'ob']
otype_sel = {
    'op': 'O_Phosphate',
    'ob': 'O_Base',
}
ions = ['MG', 'NA']
forcefields = ['mMg_opc', 'mMg_tip3p', 'nMg_tip3p']

for forcefield in forcefields:
    for otype in otypes:
        fig, axarr = plt.subplots(nrows=2, ncols=2, figsize=(10, 8), sharey=True)
        for ion, axrow in zip(ions, axarr):
            for noion, ax in zip(['', '_noion'], axrow):
                path = f'{project_dir}/replex/7ez0/production/{forcefield}{noion}/rep00'
                ax = plot_site_global_occupancy(path, site=otype_sel[otype], ion=ion, ax=ax)
                if noion: ax.set_title(ax.get_title() + " (no PDB ions)")
        fig.suptitle(f'{LABELS[forcefield]}', fontsize=16)
        plt.tight_layout()
        fig.savefig(f'figures/{pdbid}_global_occupancy_{otype}_{forcefield}.pdf')


In [ ]:
pdbid = '7ez0'
otypes = ['op', 'ob']
otype_sel = {
    'op': 'O_Phosphate',
    'ob': 'O_Base',
}
ions = ['MG', 'NA']
forcefields = ['mMg_opc', 'mMg_tip3p', 'nMg_tip3p']

for forcefield in forcefields:
    for otype in otypes:
        fig, axarr = plt.subplots(nrows=2, ncols=2, figsize=(10, 8), sharey=True)
        for ion, axrow in zip(ions, axarr):
            for noion, ax in zip(['', '_noion'], axrow):
                path = f'{project_dir}/replex/7ez0/production/{forcefield}{noion}/rep00'
                ax = plot_site_occupancy(path, site=otype_sel[otype], ion=ion, ax=ax)
                if noion: ax.set_title(ax.get_title() + " (no PDB ions)")
        fig.suptitle(f'{LABELS[forcefield]}', fontsize=16)
        plt.tight_layout()
        fig.savefig(f'figures/{pdbid}_occupancy_{otype}_{forcefield}.pdf')